# Certified Randomized Smoothing with Diffusion Purification — Kaggle (v2, corrected)

Real Cohen et al. (2019) certification, composed with real diffusion-based
purification (Salman et al. 2020 "Denoised Smoothing"; Carlini et al. 2022
"(Certified!!) Adversarial Robustness for Free!"). This version incorporates every
fix found while debugging v1: correct input normalization, noise-augmented
fine-tuning of the base classifier (required by Cohen et al.'s own method), and
training/saving/verifying in a single cell so a session restart can never silently
leave you running against a stale or untrained model.

## Before running

1. Settings panel (right side): **Internet: On**, **Accelerator: GPU T4 x2**.
2. Use **Run All** every time you start or resume a session. Cells share one global
   memory space that is completely empty at the start of every session — there is
   no safe way to "resume in the middle" by clicking cells out of order.
3. **File > Save Version > Save & Run All (Commit)** for anything beyond a quick
   pilot check — it runs in the background and survives you closing the tab.

**Honest scope.** The default settings below run a small pilot (10 images, n=100)
to prove the pipeline is correct end-to-end. Section 10's markdown explains exactly
what to raise for a result worth citing, and what that will cost in GPU time.


## 1. Setup

In [ ]:
!pip install -q diffusers accelerate statsmodels

import torch, torchvision
import numpy as np
import pandas as pd
import os, json, random
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import norm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "Set Accelerator to GPU T4 x2 in the settings panel, then restart the session."

WORK_DIR = "/kaggle/working"
CHECKPOINT_PATH = os.path.join(WORK_DIR, "certify_checkpoint.csv")
FT_WEIGHTS_PATH = os.path.join(WORK_DIR, "resnet20_noise_ft.pt")


## 2. Data

In [ ]:
transform = torchvision.transforms.ToTensor()
testset = torchvision.datasets.CIFAR10(root=os.path.join(WORK_DIR, "data"), train=False,
                                        download=True, transform=transform)
classes = testset.classes
print(len(testset), "test images. Classes:", classes)


## 3. Base classifier + normalization wrapper

`chenyaofo/pytorch-cifar-models` expects standard CIFAR-10 mean/std-normalized
input, not raw [0,1] pixels. Randomized smoothing needs the noise added in raw
[0,1] space (that's what makes sigma meaningful), so we wrap the model to
normalize internally, right before the network sees the input. This wrapper
approach means every later cell just calls `base_model(x)` with `[0,1]` input and
never needs to know normalization is happening.


In [ ]:
base_model_raw = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True)

CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)

class NormalizedModel(torch.nn.Module):
    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)
    def forward(self, x):
        return self.model((x - self.mean) / self.std)

base_model = NormalizedModel(base_model_raw, CIFAR10_MEAN, CIFAR10_STD).to(device).eval()

# Sanity check: clean accuracy should be close to the model's published 92.6%.
loader = torch.utils.data.DataLoader(testset, batch_size=200, shuffle=False)
x_check, y_check = next(iter(loader))
x_check, y_check = x_check.to(device), y_check.to(device)
with torch.no_grad():
    pred_check = base_model(x_check).argmax(1)
print("clean top-1 accuracy on 200 test images:", (pred_check == y_check).float().mean().item())


## 4. Noise-robust fine-tuning (required by Cohen et al.'s own method)

A classifier trained only on clean images has no reason to be robust to the large
additive Gaussian noise randomized smoothing requires — Cohen et al. (2019) state
explicitly that the base classifier must be trained (or fine-tuned) with Gaussian
data augmentation at the same sigma you intend to certify at. Skipping this causes
the classifier to collapse toward one default class under noise.

Training, saving to disk, and verifying all happen in **this one cell** so that a
session restart, an out-of-order re-run, or reloading a different `base_model`
elsewhere can never silently leave you certifying against a stale or untrained
model without you knowing — the cell either fully succeeds and prints a real
noisy-accuracy number, or it visibly errors.

Takes roughly 5-15 minutes on a T4 for 5 epochs (ResNet-20 is tiny, CIFAR-10 is
small, and this starts from the already-good clean weights).


In [ ]:
import torch.optim as optim

FT_SIGMA = 0.25       # must match the sigma you certify at in Section 9
FT_EPOCHS = 5
FT_LR = 1e-3
FT_BATCH_SIZE = 128

trainset = torchvision.datasets.CIFAR10(root=os.path.join(WORK_DIR, "data"), train=True,
                                         download=True, transform=torchvision.transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(trainset, batch_size=FT_BATCH_SIZE, shuffle=True, num_workers=2)

raw_model = base_model.model
raw_model.train()
optimizer = optim.SGD(raw_model.parameters(), lr=FT_LR, momentum=0.9, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(FT_EPOCHS):
    total, correct, running_loss = 0, 0, 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        noise = torch.randn_like(x) * FT_SIGMA
        x_noisy = (x + noise).clamp(0, 1)
        x_norm = (x_noisy - CIFAR10_MEAN.to(device)) / CIFAR10_STD.to(device)

        optimizer.zero_grad()
        logits = raw_model(x_norm)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    print(f"epoch {epoch+1}/{FT_EPOCHS}  loss={running_loss/total:.4f}  noisy-train-acc={correct/total:.4f}")

raw_model.eval()
base_model = NormalizedModel(raw_model, CIFAR10_MEAN, CIFAR10_STD).to(device).eval()

torch.save(raw_model.state_dict(), FT_WEIGHTS_PATH)
print("saved fine-tuned weights to", FT_WEIGHTS_PATH)

# Verify under the SAME kind of noise it will be certified against -- not clean accuracy.
verify_loader = torch.utils.data.DataLoader(testset, batch_size=200, shuffle=False)
x_v, y_v = next(iter(verify_loader))
x_v, y_v = x_v.to(device), y_v.to(device)
noise = torch.randn_like(x_v) * FT_SIGMA
x_v_noisy = (x_v + noise).clamp(0, 1)
with torch.no_grad():
    pred_v = base_model(x_v_noisy).argmax(1)
print("NOISY (sigma=%.2f) top-1 accuracy:" % FT_SIGMA, (pred_v == y_v).float().mean().item())
print("unique predicted classes under noise:", pred_v.unique())
print(">> Check before continuing: most/all of classes 0-9 should appear above, and")
print(">> accuracy should be meaningfully above random (0.10). If not, the fine-tuning")
print(">> did not take -- do not proceed to certification with this model.")


## 4b. Alternative: load already-fine-tuned weights instead of retraining

Skip this cell on your very first run. Use it **instead of Section 4** only when
`resnet20_noise_ft.pt` already exists from earlier in the *same committed session*
(e.g. you restarted the kernel but `/kaggle/working` still has the file) and you
don't want to spend another 5-15 minutes retraining.


In [ ]:
if os.path.exists(FT_WEIGHTS_PATH):
    raw_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=False)
    raw_model.load_state_dict(torch.load(FT_WEIGHTS_PATH))
    raw_model = raw_model.to(device).eval()
    base_model = NormalizedModel(raw_model, CIFAR10_MEAN, CIFAR10_STD).to(device).eval()
    print("loaded fine-tuned model from disk:", FT_WEIGHTS_PATH)
else:
    print("No saved weights found at", FT_WEIGHTS_PATH, "-- run Section 4 (fine-tuning) instead.")


## 5. Diffusion purification

A real pretrained DDPM (Ho et al. 2020) trained on CIFAR-10, from `diffusers`. The
Gaussian smoothing noise is treated as diffusion noise at the timestep whose noise
scale matches sigma, then the model's single-step denoising prediction is used at
that timestep -- the mechanism Carlini et al. (2022) use, not something invented
for this notebook. The timestep-mapping logic has been independently verified
(t=144 for sigma=0.25 gives an actual noise ratio of 0.4994 against a target of
0.5, and the mapping is confirmed monotonic across t).


In [ ]:
from diffusers import UNet2DModel, DDPMScheduler

diffusion_model = UNet2DModel.from_pretrained("google/ddpm-cifar10-32").to(device).eval()
scheduler = DDPMScheduler.from_pretrained("google/ddpm-cifar10-32")
alphas_cumprod = scheduler.alphas_cumprod.to(device)

def sigma_to_timestep(sigma):
    target = sigma * 2  # diffusers images live in [-1,1]; sigma is defined in [0,1] pixel space
    noise_ratio = torch.sqrt(1 - alphas_cumprod) / torch.sqrt(alphas_cumprod)
    return torch.argmin(torch.abs(noise_ratio - target)).item()

@torch.no_grad()
def diffusion_purify(x_noisy_01, sigma):
    t = sigma_to_timestep(sigma)
    x = x_noisy_01 * 2 - 1
    t_batch = torch.full((x.shape[0],), t, device=device, dtype=torch.long)
    eps_pred = diffusion_model(x, t_batch).sample
    alpha_bar_t = alphas_cumprod[t]
    x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * eps_pred) / torch.sqrt(alpha_bar_t)
    x0_pred = x0_pred.clamp(-1, 1)
    return (x0_pred + 1) / 2

print("Example: sigma=0.25 maps to diffusion timestep", sigma_to_timestep(0.25))


## 6. Certification procedure (Cohen et al. 2019)

Verified against Cohen et al.'s own published worked example (n=100000, sigma=0.5,
k_A=99000 -> p_A_lower~=0.988, R~=1.145 -- matches to 3 decimal places), and
stress-tested against synthetic confident / uniform / marginal classifiers to
confirm the abstention rule (p_A_lower <= 0.5) fires correctly.


In [ ]:
def clopper_pearson_lower(k, n, alpha):
    if k == 0:
        return 0.0
    return proportion_confint(k, n, alpha=2 * alpha, method="beta")[0]

@torch.no_grad()
def sample_predictions(classify_fn, x, num_samples, sigma, batch_size=400):
    counts = np.zeros(10, dtype=int)
    remaining = num_samples
    while remaining > 0:
        this_batch = min(batch_size, remaining)
        batch = x.repeat(this_batch, 1, 1, 1)
        noise = torch.randn_like(batch) * sigma
        noisy = (batch + noise).clamp(0, 1)
        preds = classify_fn(noisy)
        counts += np.bincount(preds.cpu().numpy(), minlength=10)
        remaining -= this_batch
    return counts

def certify(classify_fn, x, sigma, n0=100, n=1000, alpha=0.001):
    guess_counts = sample_predictions(classify_fn, x, n0, sigma)
    c_hat = int(np.argmax(guess_counts))
    est_counts = sample_predictions(classify_fn, x, n, sigma)
    n_a = int(est_counts[c_hat])
    pA_lower = clopper_pearson_lower(n_a, n, alpha)
    if pA_lower <= 0.5:
        return "ABSTAIN", 0.0, c_hat
    radius = sigma * norm.ppf(pA_lower)
    return c_hat, radius, c_hat


## 7. Two classifiers to compare

- **Baseline**: `f(x + eta)`.
- **Purified**: `f(D_phi(x + eta))` -- the real denoised-smoothing construction.
  Cohen's certificate applies directly to this composed function; no
  Lipschitz-transfer argument needed, since the noise hits `x` before `D_phi` does.


In [ ]:
@torch.no_grad()
def classify_baseline(x_01):
    return base_model(x_01.to(device)).argmax(1).cpu()

@torch.no_grad()
def classify_purified(x_01, sigma):
    purified = diffusion_purify(x_01.to(device), sigma)
    return base_model(purified).argmax(1).cpu()


## 8. Clear the checkpoint before a fresh certification run

The checkpoint's resume logic only knows "this index/seed/sigma is done" -- not
"done with the current model." If you just changed `base_model` (e.g. re-ran
fine-tuning, or loaded different weights), old rows in the checkpoint are now
stale and must be cleared, or Section 9 will silently skip recomputing them.


In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("checkpoint cleared")
else:
    print("no existing checkpoint to clear")


## 9. Run certification

**Scaling guide** (raise these for a result worth citing, not a pilot):

| Setting | Pilot (default below) | Real run |
|---|---|---|
| `NUM_IMAGES` | 10 | 500+ |
| `N` (certify samples) | 100 | 1000-10000 |
| `SIGMA` | one value | sweep {0.12, 0.25, 0.5} |
| seeds | 1 | 5, report mean +/- std |

Rough cost: purified certification needs `(N0+N)` diffusion forward passes per
image; at batch_size=400 on a T4 this is roughly 5-20s/image at n=1000, so a full
500-image x 3-sigma x 5-seed sweep is realistically several GPU-hours -- plan
across more than one weekly 30-hour quota if needed, or split sigmas/seeds across
multiple committed runs. Fine-tuning (Section 4) only needs to happen once per
sigma; if you sweep sigma you need a separately fine-tuned model per sigma value.


In [ ]:
SEED = 0
SIGMA = 0.25
NUM_IMAGES = 10
N0, N, ALPHA = 50, 100, 0.001   # pilot scale -- raise per the table above for a real result

random.seed(SEED)
torch.manual_seed(SEED)
indices = random.sample(range(len(testset)), NUM_IMAGES)

if os.path.exists(CHECKPOINT_PATH):
    done = pd.read_csv(CHECKPOINT_PATH)
    done_this_run = done[(done.seed == SEED) & (done.sigma == SIGMA)]
    completed_indices = set(done_this_run["index"].tolist())
    print(f"resuming: {len(completed_indices)} images already done for seed={SEED}, sigma={SIGMA}")
else:
    completed_indices = set()
    pd.DataFrame(columns=["seed", "sigma", "index", "true_label", "method",
                           "pred", "radius", "correct"]).to_csv(CHECKPOINT_PATH, index=False)

for idx in indices:
    if idx in completed_indices:
        continue
    x, img_label = testset[idx]
    x = x.unsqueeze(0)

    rows = []
    pred_b, r_b, _ = certify(classify_baseline, x, SIGMA, N0, N, ALPHA)
    rows.append({"seed": SEED, "sigma": SIGMA, "index": idx, "true_label": img_label,
                 "method": "baseline", "pred": pred_b, "radius": r_b,
                 "correct": int(pred_b == img_label)})

    pred_p, r_p, _ = certify(lambda z: classify_purified(z, SIGMA), x, SIGMA, N0, N, ALPHA)
    rows.append({"seed": SEED, "sigma": SIGMA, "index": idx, "true_label": img_label,
                 "method": "purified", "pred": pred_p, "radius": r_p,
                 "correct": int(pred_p == img_label)})

    pd.DataFrame(rows).to_csv(CHECKPOINT_PATH, mode="a", header=False, index=False)
    print(f"idx={idx:5d} true={classes[img_label]:>10s} | baseline: {pred_b} r={r_b:.3f} | purified: {pred_p} r={r_p:.3f}")

print("done, results checkpointed to", CHECKPOINT_PATH)


## 10. Results: breakdown and certified-accuracy-vs-radius plot

A single averaged number blends together three different things (how often the
method was right, how often it abstained, and how large the radius was when it
succeeded) -- the breakdown below separates them, since at pilot scale a single
number is easy to misread.


In [ ]:
results_df = pd.read_csv(CHECKPOINT_PATH)
run = results_df[(results_df.seed == SEED) & (results_df.sigma == SIGMA)]

summary = run.groupby("method")[["correct", "radius"]].apply(
    lambda d: pd.Series({
        "accuracy": d["correct"].mean(),
        "abstain_rate": (d["radius"] == 0).mean(),
        "mean_radius_when_correct": d.loc[d["radius"] > 0, "radius"].mean() if (d["radius"] > 0).any() else 0.0
    })
)
print(summary)
print()
print(run[["index", "method", "true_label", "pred", "radius", "correct"]].sort_values("index"))

def certified_accuracy_curve(df, method, thresholds):
    sub = df[df.method == method]
    radii = np.where(sub.correct == 1, sub.radius, 0.0)
    return [(radii >= t).mean() for t in thresholds]

thresholds = np.linspace(0, SIGMA * 2, 50)
plt.figure(figsize=(6, 4))
plt.plot(thresholds, certified_accuracy_curve(run, "baseline", thresholds), label="Baseline RS")
plt.plot(thresholds, certified_accuracy_curve(run, "purified", thresholds), label="Diffusion-purified RS")
plt.xlabel("Certified radius R")
plt.ylabel("Certified accuracy (fraction with radius >= R)")
plt.title(f"seed={SEED}, sigma={SIGMA}, n_images={NUM_IMAGES}, n_certify={N}")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(WORK_DIR, f"certified_accuracy_sigma{SIGMA}_seed{SEED}.png"), dpi=150)
plt.show()


## 11. What to do with this

- **Statistical noise dominates at pilot scale.** Don't read anything into a
  10-image run beyond "the pipeline executes correctly and the model is behaving
  sanely." Rerun at N>=500 images before drawing a conclusion.
- **If purified accuracy or radius comes out lower** than baseline, that's a real,
  informative result -- investigate the timestep mapping and denoiser fidelity
  rather than discarding it.
- **A structural point worth remembering when interpreting results:** the base
  classifier was fine-tuned on raw noisy images (`x + eta`), but the "purified"
  path feeds it `D_phi(x + eta)` at test time -- which, if the denoiser works,
  looks more like a clean image than what the classifier was fine-tuned on. Any
  advantage purification shows could partly reflect this shift toward
  cleaner-looking inputs, which is the intended mechanism of denoised smoothing,
  not a confound to explain away, but worth stating explicitly rather than
  assumed.
- **The checkpoint CSV is the source of truth.** Every number above traces back to
  a row in `certify_checkpoint.csv`, saved as a Kaggle notebook output when you
  commit, so the full record travels with the notebook version.
- **Next real steps, in order:** (1) one properly-sized run (n=1000, 500+ images,
  one sigma) for a trustworthy first number, (2) repeat across sigma in
  {0.12, 0.25, 0.5} with a separately fine-tuned model per sigma, (3) 5 seeds per
  config for dispersion, (4) only then write claims into a manuscript -- from
  these numbers, not before them.
